# Le cycle complet de test fonctionnel assisté par IA
## Du cahier des charges à la livraison — fil rouge : Panier e-commerce

**Formation** — Digital House Company · Joël Parfait Kuate · hello@dhcompany.pro

Ce notebook parcourt, document par document, l'intégralité d'un projet de test fonctionnel — de l'analyse du besoin jusqu'à la livraison — en suivant la cartographie de référence (voir le document *Cartographie des documents de test*).

Pour chaque document :
1. Sa définition et sa place dans le cycle (qui le produit, quand).
2. Le **prompt structuré** (Rôle·Tâche·Contexte·Format·Contraintes) utilisé pour le générer avec une IA.
3. Une **production exécutée** — soit le document généré (texte), soit un artefact exécutable qui en démontre la mécanique (tableau, fonction, test).

> Toutes les cellules de code ont été exécutées. Les exemples de réponse IA sont rédigés pour illustrer fidèlement le type de sortie obtenue, à valider et adapter à votre contexte réel.

---
# Phase 0 — Cadrage et analyse des besoins

Avant tout test, il faut un besoin clair. Cette phase produit les premiers maillons de la chaîne de traçabilité.

## 0.1 — Cahier des charges / Spécifications fonctionnelles

**Qui :** Maîtrise d'ouvrage / Product Owner — relu et challengé par le testeur.
**Quand :** Avant le développement.

### Prompt structuré

```
Rôle : analyste QA spécialisé en revue de spécifications.
Tâche : relis cet extrait de cahier des charges et identifie les ambiguïtés, contradictions et besoins implicites non formulés.
Contexte : extrait ci-dessous, fonctionnalité « panier d'achat » d'un site e-commerce.
Format : liste numérotée, une ambiguïté par ligne, avec la question à poser au métier pour la lever.
Contraintes : ne pas reformuler le cahier des charges, uniquement signaler les points flous.
```

In [1]:
extrait_cahier_des_charges = """
Le client doit pouvoir ajouter un produit à son panier depuis la fiche produit.
Le panier affiche le total. Un code promo peut être appliqué. Si le produit n'est
plus disponible, le système doit en informer l'utilisateur. La quantité maximale
est limitée par le stock.
"""

# Exemple de sortie obtenue avec un assistant IA, à partir du prompt structuré ci-dessus.
ambiguites_detectees = [
    ("Quantité par défaut à l'ajout", "Si l'utilisateur ne précise rien, ajoute-t-on 1 unité par défaut ?"),
    ("Définition de « plus disponible »", "Stock à 0 uniquement, ou aussi produit désactivé/retiré du catalogue ?"),
    ("Type de codes promo", "Un seul code cumulable, ou plusieurs codes combinables sur la même commande ?"),
    ("Comportement à la limite de stock", "Le système bloque-t-il l'ajout, ou propose-t-il la quantité maximale disponible ?"),
    ("Persistance du panier", "Le panier est-il conservé après déconnexion de l'utilisateur, et pendant combien de temps ?"),
]

print(f"{len(ambiguites_detectees)} ambiguïtés détectées dans l'extrait :\n")
for i, (titre, question) in enumerate(ambiguites_detectees, 1):
    print(f"{i}. {titre}")
    print(f"   → Question au métier : {question}\n")

5 ambiguïtés détectées dans l'extrait :

1. Quantité par défaut à l'ajout
   → Question au métier : Si l'utilisateur ne précise rien, ajoute-t-on 1 unité par défaut ?

2. Définition de « plus disponible »
   → Question au métier : Stock à 0 uniquement, ou aussi produit désactivé/retiré du catalogue ?

3. Type de codes promo
   → Question au métier : Un seul code cumulable, ou plusieurs codes combinables sur la même commande ?

4. Comportement à la limite de stock
   → Question au métier : Le système bloque-t-il l'ajout, ou propose-t-il la quantité maximale disponible ?

5. Persistance du panier
   → Question au métier : Le panier est-il conservé après déconnexion de l'utilisateur, et pendant combien de temps ?



**Constat pédagogique :** ces 5 questions sont exactement celles qu'une revue manuelle, faite sous pression de planning, oublie le plus souvent de poser. L'IA ne remplace pas l'échange avec le métier — elle évite d'arriver à cet échange les mains vides.

## 0.2 — User story et critère d'acceptation (Gherkin)

**Qui :** Product Owner, avec contribution du testeur.
**Quand :** Dès la user story rédigée.

### Prompt structuré

```
Rôle : analyste QA, spécialiste recette métier.
Tâche : rédige les critères d'acceptation au format Gherkin pour la user story ci-dessous.
Contexte : « En tant que client, je veux appliquer un code promo à mon panier pour bénéficier d'une remise. »
Format : critères Gherkin (Étant donné / Quand / Alors), 4 à 5 critères.
Contraintes : inclure le cas du code promo invalide et celui du code promo déjà utilisé.
```

In [2]:
user_story = "En tant que client, je veux appliquer un code promo à mon panier pour bénéficier d'une remise."

criteres_gherkin = """\
Scénario 1 — Code promo valide
  Étant donné un panier contenant des articles
  Quand le client saisit un code promo valide
  Alors le total du panier est recalculé avec la remise appliquée

Scénario 2 — Code promo invalide
  Étant donné un panier contenant des articles
  Quand le client saisit un code promo inexistant
  Alors un message d'erreur explicite est affiché et le total reste inchangé

Scénario 3 — Code promo expiré
  Étant donné un code promo dont la date de validité est dépassée
  Quand le client tente de l'appliquer
  Alors le système refuse le code et précise qu'il est expiré

Scénario 4 — Code promo déjà utilisé par ce client
  Étant donné un code promo à usage unique déjà consommé par ce client
  Quand le client tente de le réappliquer
  Alors le système refuse et indique que le code a déjà été utilisé
"""
print(f"User story : {user_story}\n")
print(criteres_gherkin)

User story : En tant que client, je veux appliquer un code promo à mon panier pour bénéficier d'une remise.

Scénario 1 — Code promo valide
  Étant donné un panier contenant des articles
  Quand le client saisit un code promo valide
  Alors le total du panier est recalculé avec la remise appliquée

Scénario 2 — Code promo invalide
  Étant donné un panier contenant des articles
  Quand le client saisit un code promo inexistant
  Alors un message d'erreur explicite est affiché et le total reste inchangé

Scénario 3 — Code promo expiré
  Étant donné un code promo dont la date de validité est dépassée
  Quand le client tente de l'appliquer
  Alors le système refuse le code et précise qu'il est expiré

Scénario 4 — Code promo déjà utilisé par ce client
  Étant donné un code promo à usage unique déjà consommé par ce client
  Quand le client tente de le réappliquer
  Alors le système refuse et indique que le code a déjà été utilisé



## 0.3 — Matrice de traçabilité (RTM) — version initiale

**Qui :** Testeur / Test Analyst.
**Quand :** Dès les premières exigences stabilisées, puis mise à jour en continu.

C'est le document pivot de toute la chaîne : chaque ligne ajoutée dans les phases suivantes viendra l'enrichir.

In [3]:
import pandas as pd

rtm = pd.DataFrame([
    {"ID_Exigence": "EXI-01", "Exigence": "Ajouter un produit au panier", "Critère Gherkin": "—", "Cas de test": "—", "Statut exécution": "À faire"},
    {"ID_Exigence": "EXI-02", "Exigence": "Appliquer un code promo",      "Critère Gherkin": "—", "Cas de test": "—", "Statut exécution": "À faire"},
    {"ID_Exigence": "EXI-03", "Exigence": "Refuser un produit en rupture", "Critère Gherkin": "—", "Cas de test": "—", "Statut exécution": "À faire"},
])
rtm

,ID_Exigence,Exigence,Critère Gherkin,Cas de test,Statut exécution
0,EXI-01,Ajouter un produit au panier,—,—,À faire
1,EXI-02,Appliquer un code promo,—,—,À faire
2,EXI-03,Refuser un produit en rupture,—,—,À faire


**Constat :** à ce stade, la RTM n'est qu'un squelette — exigences identifiées, rien d'autre. Elle se remplira à chaque phase suivante. C'est cette même table que nous retrouverons, complétée, en phase C.

---
# Phase A — Préparation des tests

Trois niveaux de planification : la stratégie (cadre général), le plan (déclinaison projet), puis la conception détaillée.

## A.1 — Stratégie de test

**Qui :** Test Manager / responsable QA. **Quand :** une fois, au niveau organisation ou programme.

### Prompt structuré

```
Rôle : Test Manager.
Tâche : propose une stratégie de test priorisée par niveau de risque pour le module Panier.
Contexte : 3 fonctionnalités principales (ajout au panier, code promo, paiement) ; le paiement touche directement le revenu.
Format : tableau fonctionnalité / niveau de risque / profondeur de test recommandée.
Contraintes : justifie chaque niveau de risque en une phrase.
```

In [4]:
strategie = pd.DataFrame([
    {"Fonctionnalité": "Paiement",            "Risque": "Élevé",  "Profondeur recommandée": "Tests exhaustifs + intégration système + recette manuelle",
     "Justification": "Impact direct sur le revenu et la confiance client en cas d'échec."},
    {"Fonctionnalité": "Ajout au panier",      "Risque": "Moyen",  "Profondeur recommandée": "Tests automatisés systématiques (composant + système)",
     "Justification": "Fonctionnalité cœur, très sollicitée, mais sans impact financier direct si un bug est détecté tôt."},
    {"Fonctionnalité": "Code promo",           "Risque": "Moyen",  "Profondeur recommandée": "Tests ciblés sur les cas limites (expiration, usage unique)",
     "Justification": "Risque financier indirect (perte de marge) si la règle métier est mal appliquée."},
])
strategie

,Fonctionnalité,Risque,Profondeur recommandée,Justification
0,Paiement,Élevé,Tests exhaustifs + intégration système + recet...,Impact direct sur le revenu et la confiance cl...
1,Ajout au panier,Moyen,Tests automatisés systématiques (composant + s...,"Fonctionnalité cœur, très sollicitée, mais san..."
2,Code promo,Moyen,"Tests ciblés sur les cas limites (expiration, ...",Risque financier indirect (perte de marge) si ...


## A.2 — Plan de test (structure IEEE 829)

**Qui :** Test Manager / Test Lead. **Quand :** au lancement du projet.

### Prompt structuré

```
Rôle : Test Manager.
Tâche : génère un squelette de plan de test conforme à la structure IEEE 829.
Contexte : module Panier d'un site e-commerce, cycle de 2 semaines, équipe de 2 testeurs.
Format : sections IEEE 829 standard, chacune avec 1-2 phrases de contenu adapté au contexte.
Contraintes : reste concis, ce squelette sera complété manuellement par la suite.
```

In [5]:
def generer_squelette_plan_de_test(module, duree, equipe):
    sections = {
        "1. Identifiant du plan de test": f"PT-{module.upper()}-2026-01",
        "2. Références": "Cahier des charges v1.2, user stories EXI-01 à EXI-03, RTM initiale.",
        "3. Introduction": f"Plan de test pour le module {module}, périmètre e-commerce.",
        "4. Éléments à tester": f"Module {module} : ajout, retrait, calcul du total, code promo.",
        "5. Risques logiciel": "Concurrence d'accès au panier sur sessions multiples ; arrondi monétaire.",
        "6. Caractéristiques à tester": "Ajout/retrait d'articles, calcul du total, application de remise.",
        "7. Caractéristiques à ne pas tester": "Recommandations produit, historique d'achats (hors périmètre).",
        "8. Approche": "Tests composants automatisés + tests d'intégration + 1 session de recette manuelle.",
        "9. Critères de réussite/échec": "100% des cas critiques passés, 0 anomalie bloquante ouverte.",
        "10. Critères de suspension/reprise": "Suspension si l'environnement de test est indisponible plus de 4h.",
        "11. Livrables de test": "Cas de test, rapport d'exécution, rapport de synthèse, RTM finale.",
        "12. Tâches restantes": "Aucune à ce stade (début de cycle).",
        "13. Besoins d'environnement": "Environnement de recette avec API de paiement en mode sandbox.",
        "14. Besoins en personnel et formation": f"Équipe de {equipe} testeurs, formés à l'outil de gestion de tests.",
        "15. Responsabilités": "Le Test Lead valide les cas de test ; les testeurs exécutent et remontent les anomalies.",
        "16. Calendrier": f"Durée totale : {duree}. Conception semaine 1, exécution semaine 2.",
        "17. Risques de planification": "Dépendance à la disponibilité de l'environnement de recette.",
        "18. Approbations": "Test Manager, Product Owner.",
    }
    return sections

plan = generer_squelette_plan_de_test("Panier", "2 semaines", 2)
for section, contenu in plan.items():
    print(f"{section}\n  {contenu}\n")

1. Identifiant du plan de test
  PT-PANIER-2026-01

2. Références
  Cahier des charges v1.2, user stories EXI-01 à EXI-03, RTM initiale.

3. Introduction
  Plan de test pour le module Panier, périmètre e-commerce.

4. Éléments à tester
  Module Panier : ajout, retrait, calcul du total, code promo.

5. Risques logiciel
  Concurrence d'accès au panier sur sessions multiples ; arrondi monétaire.

6. Caractéristiques à tester
  Ajout/retrait d'articles, calcul du total, application de remise.

7. Caractéristiques à ne pas tester
  Recommandations produit, historique d'achats (hors périmètre).

8. Approche
  Tests composants automatisés + tests d'intégration + 1 session de recette manuelle.

9. Critères de réussite/échec
  100% des cas critiques passés, 0 anomalie bloquante ouverte.

10. Critères de suspension/reprise
  Suspension si l'environnement de test est indisponible plus de 4h.

11. Livrables de test
  Cas de test, rapport d'exécution, rapport de synthèse, RTM finale.

12. Tâches re

## A.3 — Spécification de conception des tests (Test Design Specification)

**Qui :** Test Analyst. **Quand :** après stabilisation des exigences d'un lot.

### Prompt structuré

```
Rôle : Test Analyst.
Tâche : identifie les conditions de test pour la fonction calculerTotal(prix, quantité, code_promo) par technique de conception.
Contexte : prix > 0, quantité entre 1 et le stock disponible, code_promo optionnel.
Format : tableau technique / condition de test.
Contraintes : couvre au minimum partitions d'équivalence et valeurs limites.
```

In [6]:
conditions_de_test = pd.DataFrame([
    {"Technique": "Partition d'équivalence", "Condition de test": "Quantité dans la plage valide (ex : 1 à stock)"},
    {"Technique": "Partition d'équivalence", "Condition de test": "Quantité hors plage (négative, zéro, supérieure au stock)"},
    {"Technique": "Valeur limite",            "Condition de test": "Quantité = 1 (limite basse)"},
    {"Technique": "Valeur limite",            "Condition de test": "Quantité = stock exact (limite haute)"},
    {"Technique": "Valeur limite",            "Condition de test": "Quantité = stock + 1 (juste au-delà de la limite)"},
    {"Technique": "Partition d'équivalence", "Condition de test": "Code promo absent / valide / invalide / expiré"},
])
conditions_de_test

,Technique,Condition de test
0,Partition d'équivalence,Quantité dans la plage valide (ex : 1 à stock)
1,Partition d'équivalence,"Quantité hors plage (négative, zéro, supérieur..."
2,Valeur limite,Quantité = 1 (limite basse)
3,Valeur limite,Quantité = stock exact (limite haute)
4,Valeur limite,Quantité = stock + 1 (juste au-delà de la limite)
5,Partition d'équivalence,Code promo absent / valide / invalide / expiré


## A.4 — Spécification des cas de test (Test Case Specification)

**Qui :** Testeur. **Quand :** en continu, lot par lot.

### Prompt structuré

```
Rôle : testeur QA senior.
Tâche : transforme ces conditions de test en cas de test détaillés exécutables.
Contexte : fonction calculerTotal(prix, quantité, code_promo) du module Panier.
Format : tableau ID / préconditions / entrée / résultat attendu.
Contraintes : un cas de test par condition de test identifiée précédemment.
```

In [7]:
cas_de_test = pd.DataFrame([
    {"ID": "CT-01", "Préconditions": "Article en stock (10 unités)", "Entrée": "quantité=1",  "Résultat attendu": "Ajout réussi, total = prix unitaire"},
    {"ID": "CT-02", "Préconditions": "Article en stock (10 unités)", "Entrée": "quantité=-1", "Résultat attendu": "Erreur : quantité invalide"},
    {"ID": "CT-03", "Préconditions": "Article en stock (10 unités)", "Entrée": "quantité=10", "Résultat attendu": "Ajout réussi (limite haute du stock)"},
    {"ID": "CT-04", "Préconditions": "Article en stock (10 unités)", "Entrée": "quantité=11", "Résultat attendu": "Erreur : stock insuffisant"},
    {"ID": "CT-05", "Préconditions": "Panier non vide",              "Entrée": "code_promo='BIENVENUE10'", "Résultat attendu": "Remise de 10% appliquée au total"},
    {"ID": "CT-06", "Préconditions": "Panier non vide",              "Entrée": "code_promo='PERIME2024'",  "Résultat attendu": "Erreur : code promo expiré"},
])
cas_de_test

,ID,Préconditions,Entrée,Résultat attendu
0,CT-01,Article en stock (10 unités),quantité=1,"Ajout réussi, total = prix unitaire"
1,CT-02,Article en stock (10 unités),quantité=-1,Erreur : quantité invalide
2,CT-03,Article en stock (10 unités),quantité=10,Ajout réussi (limite haute du stock)
3,CT-04,Article en stock (10 unités),quantité=11,Erreur : stock insuffisant
4,CT-05,Panier non vide,code_promo='BIENVENUE10',Remise de 10% appliquée au total
5,CT-06,Panier non vide,code_promo='PERIME2024',Erreur : code promo expiré


## A.5 — Jeux de données de test

**Qui :** Testeur, parfois avec le DBA. **Quand :** avant l'exécution.

### Prompt structuré

```
Rôle : testeur QA.
Tâche : génère un jeu de données de test couvrant les cas limites pour les cas de test CT-01 à CT-06.
Contexte : référentiel produit du Panier (3 produits : casque audio, clavier mécanique, souris).
Format : table de données prête à charger.
Contraintes : inclure au moins un cas de stock à 0 et un prix avec décimales.
```

In [8]:
jeu_de_donnees = pd.DataFrame([
    {"produit": "casque-audio", "prix": 79.90,  "stock": 12, "quantite_test": 1,  "code_promo": None},
    {"produit": "clavier-meca", "prix": 109.00, "stock": 3,  "quantite_test": 3,  "code_promo": None},
    {"produit": "clavier-meca", "prix": 109.00, "stock": 3,  "quantite_test": 4,  "code_promo": None},
    {"produit": "souris-pro",   "prix": 39.50,  "stock": 0,  "quantite_test": 1,  "code_promo": None},
    {"produit": "casque-audio", "prix": 79.90,  "stock": 12, "quantite_test": 1,  "code_promo": "BIENVENUE10"},
    {"produit": "casque-audio", "prix": 79.90,  "stock": 12, "quantite_test": 1,  "code_promo": "PERIME2024"},
])
jeu_de_donnees

,produit,prix,stock,quantite_test,code_promo
0,casque-audio,79.9,12,1,NaN
1,clavier-meca,109.0,3,3,NaN
2,clavier-meca,109.0,3,4,NaN
3,souris-pro,39.5,0,1,NaN
4,casque-audio,79.9,12,1,BIENVENUE10
5,casque-audio,79.9,12,1,PERIME2024


## A.6 — Rapport de transmission des éléments de test

**Qui :** Développeur / Release Manager → Testeur. **Quand :** à chaque livraison de build.

### Prompt structuré

```
Rôle : assistant release management.
Tâche : synthétise ce changelog technique en langage clair pour l'équipe de test.
Contexte : liste de commits ci-dessous.
Format : 3 à 5 puces, langage métier, pas de jargon Git.
Contraintes : signale explicitement tout changement touchant au paiement.
```

In [9]:
commits_bruts = [
    "fix(panier): correction arrondi total avec code promo",
    "feat(panier): ajout limite de quantité basée sur stock réel",
    "refactor(paiement): migration vers nouvelle version API Stripe",
    "chore: mise à jour dépendances",
]

# Exemple de synthèse obtenue à partir du prompt structuré
synthese_transmission = """\
Build 2026.06.30-rc1 — prêt pour test

- Le calcul du total avec code promo a été corrigé (problème d'arrondi sur certains montants).
- Une nouvelle limite empêche désormais d'ajouter plus d'articles que le stock réel disponible.
- ATTENTION — changement sur le paiement : migration vers une nouvelle version de l'API Stripe.
  Ce point nécessite une vérification prioritaire avant toute autre exécution.
- Mise à jour technique sans impact fonctionnel (dépendances).
"""
print(synthese_transmission)

Build 2026.06.30-rc1 — prêt pour test

- Le calcul du total avec code promo a été corrigé (problème d'arrondi sur certains montants).
- Une nouvelle limite empêche désormais d'ajouter plus d'articles que le stock réel disponible.
- ATTENTION — changement sur le paiement : migration vers une nouvelle version de l'API Stripe.
  Ce point nécessite une vérification prioritaire avant toute autre exécution.
- Mise à jour technique sans impact fonctionnel (dépendances).



---
# Phase B — Exécution des tests

On exécute, on consigne, on remonte. Reprenons le code du module Panier (issu du notebook « Niveaux de test ») pour produire de vraies exécutions.

In [10]:
# Code du système sous test (rappel simplifié du notebook « Niveaux de test »)
class ProduitIndisponible(Exception):
    pass

class Catalogue:
    def __init__(self):
        self._produits = {
            "casque-audio": {"nom": "Casque Audio", "prix": 79.90, "stock": 12},
            "clavier-meca": {"nom": "Clavier mécanique", "prix": 109.00, "stock": 3},
            "souris-pro":   {"nom": "Souris ergonomique", "prix": 39.50, "stock": 0},
        }
    def get_produit(self, ref):
        if ref not in self._produits:
            raise ProduitIndisponible(f"Produit inconnu : {ref}")
        return dict(self._produits[ref])

class Panier:
    def __init__(self, catalogue):
        self.catalogue = catalogue
        self.lignes = {}
    def ajouter(self, ref, quantite):
        if quantite <= 0:
            raise ValueError("La quantité doit être strictement positive.")
        produit = self.catalogue.get_produit(ref)
        quantite_totale = self.lignes.get(ref, 0) + quantite
        if produit["stock"] < quantite_totale - 1:   # ⚠️ bug : décalage d'une unité (build 2026.06.30-rc1)
            raise ProduitIndisponible(f"Stock insuffisant pour {produit['nom']}")
        self.lignes[ref] = quantite_totale
    def calculer_total(self, code_promo=None):
        codes_valides = {"BIENVENUE10": 0.90}
        codes_expires = {"PERIME2024"}
        total = sum(self.catalogue.get_produit(ref)["prix"] * qte for ref, qte in self.lignes.items())
        if code_promo in codes_expires:
            raise ValueError("Code promo expiré.")
        if code_promo in codes_valides:
            total *= codes_valides[code_promo]
        elif code_promo is not None:
            raise ValueError("Code promo invalide.")
        return round(total, 2)

print("Module Panier chargé — prêt pour exécution des cas de test CT-01 à CT-06.")

Module Panier chargé — prêt pour exécution des cas de test CT-01 à CT-06.


## B.1 — Journal de test (Test Log)

**Qui :** Testeur (souvent généré par l'outil de gestion de tests). **Quand :** pendant chaque session d'exécution.

Plutôt que de remplir ce journal à la main, nous le **construisons automatiquement** en exécutant chaque cas de test CT-01 à CT-06 et en consignant le résultat réel.

In [11]:
import datetime

def executer_cas_de_test(id_ct, fonction):
    horodatage = datetime.datetime.now().isoformat(timespec="seconds")
    try:
        fonction()
        return {"ID": id_ct, "Horodatage": horodatage, "Résultat": "PASS", "Détail": "Conforme au résultat attendu."}
    except Exception as e:
        return {"ID": id_ct, "Horodatage": horodatage, "Résultat": "FAIL", "Détail": f"{type(e).__name__}: {e}"}

def ct_01():
    c = Catalogue(); p = Panier(c); p.ajouter("casque-audio", 1)
    assert p.calculer_total() == 79.90

def ct_02():
    c = Catalogue(); p = Panier(c)
    try:
        p.ajouter("casque-audio", -1)
        raise AssertionError("Aurait dû lever ValueError")
    except ValueError:
        pass

def ct_03():
    c = Catalogue(); p = Panier(c); p.ajouter("clavier-meca", 3)
    assert p.calculer_total() == 327.00

def ct_04():
    c = Catalogue(); p = Panier(c)
    try:
        p.ajouter("clavier-meca", 4)
        raise AssertionError("Aurait dû lever ProduitIndisponible")
    except ProduitIndisponible:
        pass

def ct_05():
    c = Catalogue(); p = Panier(c); p.ajouter("casque-audio", 1)
    assert p.calculer_total("BIENVENUE10") == round(79.90 * 0.9, 2)

def ct_06():
    c = Catalogue(); p = Panier(c); p.ajouter("casque-audio", 1)
    try:
        p.calculer_total("PERIME2024")
        raise AssertionError("Aurait dû lever ValueError")
    except ValueError:
        pass

journal = pd.DataFrame([
    executer_cas_de_test("CT-01", ct_01),
    executer_cas_de_test("CT-02", ct_02),
    executer_cas_de_test("CT-03", ct_03),
    executer_cas_de_test("CT-04", ct_04),
    executer_cas_de_test("CT-05", ct_05),
    executer_cas_de_test("CT-06", ct_06),
])
journal

,ID,Horodatage,Résultat,Détail
0,CT-01,2026-06-30T21:55:41,PASS,Conforme au résultat attendu.
1,CT-02,2026-06-30T21:55:41,PASS,Conforme au résultat attendu.
2,CT-03,2026-06-30T21:55:41,PASS,Conforme au résultat attendu.
3,CT-04,2026-06-30T21:55:41,FAIL,AssertionError: Aurait dû lever ProduitIndispo...
4,CT-05,2026-06-30T21:55:41,PASS,Conforme au résultat attendu.
5,CT-06,2026-06-30T21:55:41,PASS,Conforme au résultat attendu.


## B.2 — Rapport d'anomalie (Test Incident Report)

**Qui :** Testeur. **Quand :** dès qu'un écart est constaté.

Le journal ci-dessus (B.1) montre que **`CT-04` a échoué** : le système a accepté l'ajout de 4 claviers mécaniques alors que le stock n'en compte que 3. Voici comment l'IA aide à structurer ce constat en rapport d'anomalie complet, à partir de l'observation informelle du testeur.

### Prompt structuré

```
Rôle : testeur QA.
Tâche : structure cette observation informelle en rapport d'anomalie complet.
Contexte : observation brute ci-dessous, cas de test CT-04, module Panier, build 2026.06.30-rc1.
Format : champs standards (titre, sévérité, étapes de reproduction, attendu/obtenu, environnement).
Contraintes : propose une sévérité justifiée par l'impact métier.
```

In [12]:
observation_brute = "le panier accepte 4 claviers alors qu'il en reste que 3 en stock, total faux aussi"

# Exemple de rapport structuré généré à partir du prompt
rapport_anomalie = {
    "ID": "BUG-2026-0142",
    "Titre": "Le panier accepte une quantité supérieure au stock disponible (Clavier mécanique)",
    "Sévérité": "Majeure — impact direct sur la fiabilité du stock affiché au client et risque de survente",
    "Build concerné": "2026.06.30-rc1",
    "Cas de test associé": "CT-04",
    "Étapes de reproduction": [
        "1. Aller sur la fiche produit Clavier mécanique (stock = 3).",
        "2. Ajouter 4 unités au panier en une seule fois.",
        "3. Observer le panier.",
    ],
    "Résultat attendu": "Le système refuse l'ajout et signale que le stock disponible est de 3.",
    "Résultat obtenu": "Le système accepte l'ajout de 4 unités ; le total du panier est calculé sur 4 unités.",
    "Environnement": "Recette — build 2026.06.30-rc1",
}
for champ, valeur in rapport_anomalie.items():
    if isinstance(valeur, list):
        print(f"{champ} :")
        for ligne in valeur:
            print(f"   {ligne}")
    else:
        print(f"{champ} : {valeur}")

ID : BUG-2026-0142
Titre : Le panier accepte une quantité supérieure au stock disponible (Clavier mécanique)
Sévérité : Majeure — impact direct sur la fiabilité du stock affiché au client et risque de survente
Build concerné : 2026.06.30-rc1
Cas de test associé : CT-04
Étapes de reproduction :
   1. Aller sur la fiche produit Clavier mécanique (stock = 3).
   2. Ajouter 4 unités au panier en une seule fois.
   3. Observer le panier.
Résultat attendu : Le système refuse l'ajout et signale que le stock disponible est de 3.
Résultat obtenu : Le système accepte l'ajout de 4 unités ; le total du panier est calculé sur 4 unités.
Environnement : Recette — build 2026.06.30-rc1


## B.3 — Rapport de statut intermédiaire (Interim Test Status Report)

**Qui :** Test Manager. **Quand :** à fréquence régulière (quotidienne/hebdomadaire).

Ce rapport s'appuie directement sur le journal de test (B.1) — aucune ressaisie, uniquement une agrégation.

### Prompt structuré

```
Rôle : Test Manager.
Tâche : rédige un résumé exécutif de l'avancement des tests à partir des données d'exécution.
Contexte : tableau d'exécution ci-dessous (journal de test).
Format : 3 phrases maximum, compréhensibles par un Product Owner non technique.
Contraintes : mentionne le taux de réussite et toute anomalie bloquante.
```

In [13]:
total = len(journal)
reussis = (journal["Résultat"] == "PASS").sum()
echecs = (journal["Résultat"] == "FAIL").sum()
taux = round(reussis / total * 100, 1)

resume_executif = (
    f"Sur {total} cas de test exécutés ce jour sur le module Panier, {reussis} sont passés avec succès "
    f"({taux}% de réussite). {echecs} anomalie(s) ont été détectée(s) et documentée(s) "
    f"(voir BUG-2026-0142, sévérité majeure). Recommandation : correction de l'anomalie avant "
    f"poursuite de la recette."
)
print(resume_executif)

Sur 6 cas de test exécutés ce jour sur le module Panier, 5 sont passés avec succès (83.3% de réussite). 1 anomalie(s) ont été détectée(s) et documentée(s) (voir BUG-2026-0142, sévérité majeure). Recommandation : correction de l'anomalie avant poursuite de la recette.


---
# Phase C — Clôture des tests et livraison

On transforme la matière collectée en décision : peut-on livrer, et avec quelles réserves ?

## C.1 — Matrice de traçabilité finale (RTM)

**Qui :** Test Lead / Test Analyst. **Quand :** avant la recette finale.

Reprenons la RTM initiale de la phase 0 et complétons-la avec ce que nous avons produit depuis.

In [14]:
rtm_finale = pd.DataFrame([
    {"ID_Exigence": "EXI-01", "Exigence": "Ajouter un produit au panier",     "Critère Gherkin": "Scénario 1 (implicite)", "Cas de test": "CT-01, CT-03", "Statut exécution": "PASS"},
    {"ID_Exigence": "EXI-02", "Exigence": "Appliquer un code promo",          "Critère Gherkin": "Scénarios 1-4",          "Cas de test": "CT-05, CT-06", "Statut exécution": "PASS"},
    {"ID_Exigence": "EXI-03", "Exigence": "Refuser un produit en rupture",    "Critère Gherkin": "Scénario implicite",     "Cas de test": "CT-02, CT-04", "Statut exécution": "FAIL (BUG-2026-0142)"},
])
rtm_finale

,ID_Exigence,Exigence,Critère Gherkin,Cas de test,Statut exécution
0,EXI-01,Ajouter un produit au panier,Scénario 1 (implicite),"CT-01, CT-03",PASS
1,EXI-02,Appliquer un code promo,Scénarios 1-4,"CT-05, CT-06",PASS
2,EXI-03,Refuser un produit en rupture,Scénario implicite,"CT-02, CT-04",FAIL (BUG-2026-0142)


In [15]:
exigences_sans_couverture = rtm_finale[rtm_finale["Cas de test"] == "—"]
print(f"Exigences sans aucun cas de test associé : {len(exigences_sans_couverture)}")
print("→ La RTM finale ne révèle aucun trou de couverture, mais signale 1 exigence en échec (EXI-03).")

Exigences sans aucun cas de test associé : 0
→ La RTM finale ne révèle aucun trou de couverture, mais signale 1 exigence en échec (EXI-03).


## C.2 — Rapport de synthèse des tests (Test Summary Report)

**Qui :** Test Manager. **Quand :** à la fin de chaque cycle de test.

### Prompt structuré

```
Rôle : Test Manager.
Tâche : rédige un rapport de synthèse des tests à partir de la RTM finale et du journal d'exécution.
Contexte : données ci-dessous (RTM finale, 1 anomalie majeure ouverte).
Format : structure IEEE 829 (résumé, évaluation, recommandation go/no-go).
Contraintes : formule une recommandation claire et justifiée.
```

In [16]:
nb_exigences = len(rtm_finale)
nb_ok = (rtm_finale["Statut exécution"] == "PASS").sum()
nb_ko = nb_exigences - nb_ok

synthese = f"""\
RAPPORT DE SYNTHÈSE DES TESTS — Module Panier — Build 2026.06.30-rc1

1. Résumé
   {nb_exigences} exigences couvertes, {nb_ok} validées, {nb_ko} en échec.
   1 anomalie majeure ouverte (BUG-2026-0142) sur la gestion du stock lors de l'ajout au panier.

2. Évaluation
   Les fonctionnalités d'ajout au panier (cas nominal) et d'application de code promo sont
   conformes aux exigences. La gestion des cas limites de stock présente une régression
   bloquante qui doit être corrigée avant mise en production.

3. Recommandation
   NO-GO pour la mise en production en l'état. GO conditionnel après correction de
   BUG-2026-0142 et ré-exécution ciblée de CT-04.
"""
print(synthese)

RAPPORT DE SYNTHÈSE DES TESTS — Module Panier — Build 2026.06.30-rc1

1. Résumé
   3 exigences couvertes, 2 validées, 1 en échec.
   1 anomalie majeure ouverte (BUG-2026-0142) sur la gestion du stock lors de l'ajout au panier.

2. Évaluation
   Les fonctionnalités d'ajout au panier (cas nominal) et d'application de code promo sont
   conformes aux exigences. La gestion des cas limites de stock présente une régression
   bloquante qui doit être corrigée avant mise en production.

3. Recommandation
   NO-GO pour la mise en production en l'état. GO conditionnel après correction de
   BUG-2026-0142 et ré-exécution ciblée de CT-04.



## C.3 — Procès-verbal de recette (UAT sign-off)

**Qui :** Product Owner / client, avec le testeur. **Quand :** lors de la recette utilisateur.

### Prompt structuré

```
Rôle : analyste QA, support à la recette métier.
Tâche : génère le script de recette à dérouler avec le Product Owner pour le module Panier.
Contexte : critères Gherkin du module (voir phase 0.2).
Format : étapes numérotées, langage non technique, avec case à cocher pour chaque étape.
Contraintes : prévoir une case « Validé » / « Refusé » à la fin de chaque étape.
```

In [17]:
script_recette = """\
SCRIPT DE RECETTE — Module Panier — à dérouler avec le Product Owner

[ ] Étape 1 — Ajouter le « Casque Audio » au panier depuis la fiche produit.
    Résultat attendu : le panier affiche 1 article, total = 79,90 EUR.
    Validé ☐   Refusé ☐

[ ] Étape 2 — Appliquer le code promo « BIENVENUE10 ».
    Résultat attendu : le total passe à 71,91 EUR (remise de 10% visible).
    Validé ☐   Refusé ☐

[ ] Étape 3 — Tenter d'ajouter 4 unités de « Clavier mécanique » (stock = 3).
    Résultat attendu : le système refuse et indique le stock disponible.
    Validé ☐   Refusé ☐   (connu : en échec tant que BUG-2026-0142 n'est pas corrigé)

[ ] Étape 4 — Appliquer le code promo « PERIME2024 ».
    Résultat attendu : message clair indiquant que le code est expiré.
    Validé ☐   Refusé ☐

Signature Product Owner : ____________________   Date : ____________
"""
print(script_recette)

SCRIPT DE RECETTE — Module Panier — à dérouler avec le Product Owner

[ ] Étape 1 — Ajouter le « Casque Audio » au panier depuis la fiche produit.
    Résultat attendu : le panier affiche 1 article, total = 79,90 EUR.
    Validé ☐   Refusé ☐

[ ] Étape 2 — Appliquer le code promo « BIENVENUE10 ».
    Résultat attendu : le total passe à 71,91 EUR (remise de 10% visible).
    Validé ☐   Refusé ☐

[ ] Étape 3 — Tenter d'ajouter 4 unités de « Clavier mécanique » (stock = 3).
    Résultat attendu : le système refuse et indique le stock disponible.
    Validé ☐   Refusé ☐   (connu : en échec tant que BUG-2026-0142 n'est pas corrigé)

[ ] Étape 4 — Appliquer le code promo « PERIME2024 ».
    Résultat attendu : message clair indiquant que le code est expiré.
    Validé ☐   Refusé ☐

Signature Product Owner : ____________________   Date : ____________



## C.4 — Rapport de clôture des tests (Test Closure Report)

**Qui :** Test Manager. **Quand :** en fin de projet ou de release majeure.

### Prompt structuré

```
Rôle : Test Manager.
Tâche : rédige les enseignements (leçons apprises) de ce cycle de test.
Contexte : 1 anomalie majeure détectée tardivement sur un cas limite de stock, déjà couverte par un cas de test mais en échec d'implémentation.
Format : 3 enseignements, chacun avec une action concrète pour le prochain cycle.
Contraintes : reste factuel, pas de blâme nominatif.
```

In [18]:
lecons_apprises = """\
ENSEIGNEMENTS — Cycle de test Module Panier

1. Constat : le cas limite « quantité = stock + 1 » était bien couvert par un cas de test (CT-04),
   ce qui a permis de détecter l'anomalie avant la mise en production plutôt qu'après.
   Action : généraliser la génération systématique de cas aux limites hautes pour tout module
   manipulant un stock ou une quantité plafonnée.

2. Constat : l'anomalie n'a été détectée qu'en phase d'exécution, alors que le code source
   aurait permis de l'anticiper dès la revue de conception (Test Design Specification).
   Action : ajouter une revue croisée développeur/testeur sur les conditions de test avant le
   début de l'implémentation, pas seulement avant l'exécution.

3. Constat : la synthèse exécutive (B.3) a permis une remontée rapide et compréhensible au
   Product Owner, sans attendre le rapport de fin de cycle.
   Action : généraliser ce type de synthèse intermédiaire automatisée à tous les modules à risque élevé.
"""
print(lecons_apprises)

ENSEIGNEMENTS — Cycle de test Module Panier

1. Constat : le cas limite « quantité = stock + 1 » était bien couvert par un cas de test (CT-04),
   ce qui a permis de détecter l'anomalie avant la mise en production plutôt qu'après.
   Action : généraliser la génération systématique de cas aux limites hautes pour tout module
   manipulant un stock ou une quantité plafonnée.

2. Constat : l'anomalie n'a été détectée qu'en phase d'exécution, alors que le code source
   aurait permis de l'anticiper dès la revue de conception (Test Design Specification).
   Action : ajouter une revue croisée développeur/testeur sur les conditions de test avant le
   début de l'implémentation, pas seulement avant l'exécution.

3. Constat : la synthèse exécutive (B.3) a permis une remontée rapide et compréhensible au
   Product Owner, sans attendre le rapport de fin de cycle.
   Action : généraliser ce type de synthèse intermédiaire automatisée à tous les modules à risque élevé.



## C.5 — Dossier de livraison / Release notes

**Qui :** Test Lead, avec le Release Manager. **Quand :** à chaque mise en production.

### Prompt structuré

```
Rôle : assistant release management.
Tâche : rédige des notes de version en langage clair à partir des tickets résolus et des tests passés.
Contexte : correction de BUG-2026-0142 supposée faite et re-testée avec succès.
Format : notes de version orientées utilisateur final, 3-4 lignes.
Contraintes : aucun jargon technique.
```

In [19]:
release_notes = """\
NOTES DE VERSION — 2026.07.02

- Le panier respecte désormais correctement la disponibilité réelle du stock : il n'est plus
  possible d'ajouter plus d'articles que ce qui est disponible.
- Les codes promo sont appliqués de manière fiable, y compris la détection des codes expirés.
- Cette mise à jour corrige un comportement signalé lors de notre dernière phase de test.
"""
print(release_notes)

NOTES DE VERSION — 2026.07.02

- Le panier respecte désormais correctement la disponibilité réelle du stock : il n'est plus
  possible d'ajouter plus d'articles que ce qui est disponible.
- Les codes promo sont appliqués de manière fiable, y compris la détection des codes expirés.
- Cette mise à jour corrige un comportement signalé lors de notre dernière phase de test.



---
# Synthèse — la chaîne complète, document par document

| Phase | Document | Rôle | Construit dans ce notebook |
|---|---|---|---|
| 0 | Cahier des charges (revue) | PO / Testeur | §0.1 |
| 0 | User story + critères Gherkin | PO / Testeur | §0.2 |
| 0 | RTM initiale | Testeur | §0.3 |
| A | Stratégie de test | Test Manager | §A.1 |
| A | Plan de test (IEEE 829) | Test Manager | §A.2 |
| A | Test Design Specification | Test Analyst | §A.3 |
| A | Cas de test | Testeur | §A.4 |
| A | Jeux de données | Testeur | §A.5 |
| A | Rapport de transmission | Dev → Testeur | §A.6 |
| B | Journal de test | Testeur (exécuté) | §B.1 |
| B | Rapport d'anomalie | Testeur | §B.2 |
| B | Rapport de statut intermédiaire | Test Manager | §B.3 |
| C | RTM finale | Test Lead | §C.1 |
| C | Rapport de synthèse des tests | Test Manager | §C.2 |
| C | PV de recette (UAT) | PO / Testeur | §C.3 |
| C | Rapport de clôture | Test Manager | §C.4 |
| C | Release notes | Test Lead | §C.5 |

**17 documents, une seule chaîne de traçabilité, un seul projet.** Chaque maillon a été produit avec un prompt structuré précisant explicitement sa place dans le cycle — c'est ce qui distingue une IA utile d'une IA qui invente des réponses hors contexte.

---
## Pour aller plus loin

Ce notebook s'accompagne de deux autres ressources :

- **La bibliothèque de 100 prompts + templates de context engineering** (notebook séparé) — chaque prompt utilisé ici y est référencé avec son code, et complété par des variantes pour d'autres contextes que le module Panier.
- **Le guide « Utiliser Claude pour les tests logiciels »** — comment automatiser tout ou partie de cette chaîne avec Claude Code, Claude.ai, et les serveurs MCP (Playwright notamment) pour passer du prompt manuel à un pipeline outillé.

---
**hello@dhcompany.pro** · Digital House Company · @DigitalMaker237